# 00 — Forward probe and safety checks
This notebook **does not train**. It checks that gas accelerates the car, verifies GAE boundaries and source-level PPO safeguards. Only continue if the gate passes. The fixed action probe is **diagnostic only**, never a submitted driving policy.

In [ ]:
from pathlib import Path
import subprocess, sys, zipfile, os
from google.colab import drive, files
drive.mount('/content/drive')
REPO_URL='https://github.com/MilkySon/RL_carracing_competition.git'
BRANCH='261030'
REPO=Path('/content/RL_carracing_competition')
BASE=REPO/'experiments'/'continuous_ablation'/'baseline'
V2=Path('/content/CarRacing_Continuous_v2_1')
ROOT=Path('/content/drive/MyDrive/CarRacing_Continuous_v2_1')
if not REPO.exists():
    subprocess.run(['git','clone','--branch',BRANCH,REPO_URL,str(REPO)],check=True)
assert (BASE/'train.py').exists(), f'Baseline missing: {BASE}'
if not V2.exists():
    print('Upload CarRacing_Continuous_v2_1.zip from your computer')
    uploaded=files.upload()
    name=next((x for x in uploaded if x.endswith('.zip') and 'Continuous_v2_1' in x),None)
    assert name, 'Please upload CarRacing_Continuous_v2_1.zip'
    with zipfile.ZipFile(name) as z:z.extractall('/content')
assert (V2/'scripts'/'train_v2.py').exists()
subprocess.run([sys.executable,'-m','pip','install','-q','-r',str(BASE/'requirements.txt')],check=True)
subprocess.run([sys.executable,str(V2/'scripts'/'install_overlay.py'),'--baseline',str(BASE)],check=True)
print('Baseline:',BASE,'Overlay:',V2,'Output:',ROOT)

In [ ]:
subprocess.run([sys.executable,str(V2/'scripts'/'test_boundaries.py')],check=True)
sys.path.insert(0,str(V2/'scripts'))
from test_boundaries import verify_implementation
verify_implementation(BASE)
probe_path=ROOT/'diagnostics'/'forward_probe.csv'
subprocess.run([sys.executable,str(V2/'scripts'/'forward_probe.py'),
                '--baseline',str(BASE),'--out',str(probe_path),'--seed','1000','--steps','100'],check=True)
import pandas as pd
probe=pd.read_csv(probe_path)
display(probe)
forward=float(probe.loc[probe.scenario=='forward','mean_speed'].iloc[0])
coast=float(probe.loc[probe.scenario=='coast','mean_speed'].iloc[0])
assert forward > coast + 0.1, 'FORWARD GATE FAILED: investigate environment and action mapping before training'
print('PASS: forward action increases speed compared with coasting')